In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, HTML, HBox, Layout
from IPython.display import display

# ============================================================
# LINE SEARCH IN NEWTON / QUASI-NEWTON OPTIMIZATION
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.ls-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.ls-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.ls-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14.5px;
    line-height:1.45;
    margin-bottom:7px;
}

.ls-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:7px;
    font-size:14px;
    line-height:1.45;
}

.ls-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="ls-root">

<div class="ls-header">
Line Search — Choosing the Step Size α
</div>

<div class="ls-doc">

After a descent direction <b>d<sub>k</sub></b> has been selected, the next
optimization point is

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
x<sub>k+1</sub> = x<sub>k</sub> + α d<sub>k</sub>.
</b>
</div>

The line-search problem therefore becomes a one-dimensional minimization of

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
φ(α) = f(x<sub>k</sub> + αd<sub>k</sub>).
</b>
</div>

A useful step must produce sufficient decrease and must not stop too early
along the descent direction.

Move α and observe the position on φ(α) and the two acceptance tests.

</div>

</div>
"""))

# ============================================================
# OBJECTIVE FUNCTION
# ============================================================

def f(x):

    X = x[0]-2.0
    Y = x[1]+1.0

    return X**2+4.0*Y**2+0.5*X*Y

def grad(x):

    X = x[0]-2.0
    Y = x[1]+1.0

    return np.array([2.0*X+0.5*Y,8.0*Y+0.5*X])

# ============================================================
# CURRENT POINT AND DESCENT DIRECTION
# ============================================================

xk = np.array([-3.0,3.0])

gk = grad(xk)

dk = -gk/np.linalg.norm(gk)

rho = 0.10

sigma = 0.70

phi0 = f(xk)

slope0 = gk@dk

# ============================================================
# LINE-SEARCH CURVE
# ============================================================

alpha_axis = np.linspace(0,8,2000)

phi = np.array([f(xk+a*dk) for a in alpha_axis])

armijo = phi0+rho*alpha_axis*slope0

alpha_opt = alpha_axis[np.argmin(phi)]

# ============================================================
# CONTROLS
# ============================================================

alpha_slider = FloatSlider(value=1.0,min=0.0,max=8.0,step=0.01,description='α:',continuous_update=True,readout_format='.2f',style={'description_width':'20px'},layout=Layout(width='430px'))

info = HTML(layout=Layout(width='450px'))

controls = HBox([alpha_slider,info],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='7px 10px',margin='0 0 7px 0',align_items='center'))

# ============================================================
# FIGURE
# ============================================================

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(9.0,5.4))

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

ax1.plot(alpha_axis,phi,color='red',linewidth=1.5,label=r'$\phi(\alpha)$')

ax1.plot(alpha_axis,armijo,'--',linewidth=1.1,label='Sufficient-decrease bound')

alpha_marker = ax1.axvline(alpha_slider.value,linestyle='--',linewidth=1.1)

point_marker, = ax1.plot([],[],'o',markersize=6)

ax1.axvline(alpha_opt,linestyle=':',linewidth=1.2,label=r'Exact $\alpha_{\min}$')

ax1.set_xlim(0,8)

ax1.set_ylim(0,np.max(phi)*1.03)

ax1.set_title(r'Line-Search Function $\phi(\alpha)$')
ax1.set_xlabel(r'Step size $\alpha$')
ax1.set_ylabel(r'$\phi(\alpha)$')

ax1.grid(True,linestyle=':',alpha=0.25)

ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# OBJECTIVE FUNCTION IN THE X1-X2 PLANE
# ============================================================

xx = np.linspace(-4,4,250)

yy = np.linspace(-4,4,250)

X,Y = np.meshgrid(xx,yy)

Z = (X-2.0)**2+4.0*(Y+1.0)**2+0.5*(X-2.0)*(Y+1.0)

ax2.contour(X,Y,Z,levels=18,linewidths=0.8)

ax2.plot(xk[0],xk[1],'o',markersize=6,label=r'$x_k$')

path = xk[None,:]+alpha_axis[:,None]*dk[None,:]

ax2.plot(path[:,0],path[:,1],'--',linewidth=1.0,label='Search line')

new_point, = ax2.plot([],[],'ro',markersize=6,label=r'$x_k+\alpha d_k$')

ax2.plot(2,-1,'*',markersize=10,label='Minimum')

ax2.set_xlim(-4,4)
ax2.set_ylim(-4,4)

ax2.set_title('Motion Along the Search Direction')
ax2.set_xlabel(r'$x_1$')
ax2.set_ylabel(r'$x_2$')

ax2.grid(True,linestyle=':',alpha=0.20)

ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

plt.subplots_adjust(left=0.08,right=0.98,top=0.91,bottom=0.20,wspace=0.28)

# ============================================================
# UPDATE
# ============================================================

def update(change=None):

    alpha = alpha_slider.value

    xnew = xk+alpha*dk

    phi_alpha = f(xnew)

    slope_alpha = grad(xnew)@dk

    condition1 = phi_alpha <= phi0+rho*alpha*slope0

    condition2 = slope_alpha >= sigma*slope0

    accepted = condition1 and condition2

    alpha_marker.set_xdata([alpha,alpha])

    point_marker.set_data([alpha],[phi_alpha])

    new_point.set_data([xnew[0]],[xnew[1]])

    info.value = f"""
    <div style="font-size:13.5px;">
    φ(α) = <b>{phi_alpha:.4f}</b>
    &nbsp;&nbsp;
    α<sub>min</sub> ≈ <b>{alpha_opt:.3f}</b>
    <br>
    Decrease test: <b>{"YES" if condition1 else "NO"}</b>
    &nbsp;&nbsp;
    Slope test: <b>{"YES" if condition2 else "NO"}</b>
    &nbsp;&nbsp;
    Accepted: <b>{"YES" if accepted else "NO"}</b>
    </div>
    """

    fig.canvas.draw_idle()

alpha_slider.observe(update,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(controls)
display(fig.canvas)

update()